# Chatbot Arena: Pairwise Preference Notebook (Professional-style baseline)

This notebook keeps the same data loading approach as your current notebook, but trains a pairwise preference model inspired by common paper practice (Bradley-Terry style).

Highlights:
- Pairwise training target: probability candidate A is preferred over candidate B
- Tie-aware soft labels (tie = 0.5 in training)
- Swap consistency at inference (A/B and B/A averaged)
- Validation calibration for tie probability to optimize 3-class log loss

In [ ]:
import kagglehub
import os

try:
    from google.colab import userdata
except Exception:
    userdata = None

if userdata is not None:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True)
kagglehub.login()
path = kagglehub.competition_download(
    'llm-classification-finetuning',
    output_dir=data_dir
)

print('Path to competition files:', path)

In [ ]:
from __future__ import annotations

import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path('data')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

MAX_LENGTH = 512
VAL_SIZE = 0.1
OUTPUT_DIR = 'pairwise_pref_ckpt'
SUBMISSION_PATH = 'submission_pairwise_professional.csv'

# Small parameter bump from DistilBERT (~66M) to RoBERTa-base (~125M).
MODEL_NAME = 'roberta-base'

# Keep global batch target while controlling GPU memory via accumulation.
TARGET_EFFECTIVE_BATCH = 128
TRAIN_BATCH_GPU = 16 if torch.cuda.is_available() else 2
EVAL_BATCH_GPU = TRAIN_BATCH_GPU
NUM_EPOCHS = 3.0
LEARNING_RATE = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Model:', MODEL_NAME)
print('Per-device train batch:', TRAIN_BATCH_GPU)
print('Target effective batch:', TARGET_EFFECTIVE_BATCH)


In [ ]:
import re
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

WORD_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)
CODE_BLOCK_RE = re.compile(r"```")
BULLET_RE = re.compile(r"^\s*[-*\u2022]", re.MULTILINE)
NUMBERED_RE = re.compile(r"^\s*\d+\.", re.MULTILINE)
HEADING_RE = re.compile(r"^\s*#{1,6}\s", re.MULTILINE)

REFUSAL_PATTERNS = re.compile(
    r"\b(I cannot|I can't|I'm unable|I am unable|I'm not able|as an AI|as a language model"
    r"|I don't have (the ability|access)|unable to|not (able|capable))\b",
    re.IGNORECASE,
)
HEDGE_PATTERNS = re.compile(
    r"\b(maybe|perhaps|probably|might|could be|it depends|I think|I believe"
    r"|in my opinion|roughly|approximately|around)\b",
    re.IGNORECASE,
)


def safe_load_list(value: str) -> list[str]:
    if pd.isna(value):
        return []
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return [str(value)]
    if isinstance(parsed, list):
        return [str(x) for x in parsed]
    return [str(parsed)]


def tokenize_words(text: str) -> list[str]:
    return WORD_RE.findall(text.lower())


def lexical_overlap(a: str, b: str) -> float:
    wa, wb = set(tokenize_words(a)), set(tokenize_words(b))
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / max(1, len(wa))


def repetition_score(text: str) -> float:
    words = tokenize_words(text)
    if len(words) < 4:
        return 0.0
    grams = [tuple(words[i:i+4]) for i in range(len(words) - 3)]
    unique = len(set(grams))
    return 1.0 - unique / max(1, len(grams))


def avg_sentence_len(text: str) -> float:
    sents = re.split(r"[.!?]+", text)
    lens = [len(s.split()) for s in sents if s.strip()]
    return float(sum(lens) / max(1, len(lens)))


def build_similarity_vectorizer(train_df: pd.DataFrame, test_df: pd.DataFrame) -> TfidfVectorizer:
    corpus: list[str] = []
    for df in (train_df, test_df):
        corpus.extend(df['prompt'].fillna('').astype(str).tolist())
        corpus.extend(df['response_a'].fillna('').astype(str).tolist())
        corpus.extend(df['response_b'].fillna('').astype(str).tolist())
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3,
                         max_features=50_000, sublinear_tf=True)
    vec.fit(corpus)
    return vec


def tfidf_cosine(a: str, b: str, vec: TfidfVectorizer) -> float:
    if not a or not b:
        return 0.0
    mat = vec.transform([a, b])
    return float(sk_cosine(mat[0], mat[1])[0, 0])


def extract_features(
    prompt: str, a: str, b: str, turns: int, vec: TfidfVectorizer | None
) -> dict[str, float]:
    a_len = len(a)
    b_len = len(b)
    a_words = len(tokenize_words(a))
    b_words = len(tokenize_words(b))
    p_words = len(tokenize_words(prompt))

    sim_a = tfidf_cosine(prompt, a, vec) if vec else 0.0
    sim_b = tfidf_cosine(prompt, b, vec) if vec else 0.0

    return {
        'len_ratio': math.log1p(a_len) - math.log1p(b_len),
        'word_ratio': math.log1p(a_words) - math.log1p(b_words),
        'prompt_words': float(p_words),
        'turns': float(turns),
        'overlap_a': lexical_overlap(prompt, a),
        'overlap_b': lexical_overlap(prompt, b),
        'overlap_diff': lexical_overlap(prompt, a) - lexical_overlap(prompt, b),
        'sem_a': sim_a,
        'sem_b': sim_b,
        'sem_diff': sim_a - sim_b,
        'rep_a': repetition_score(a),
        'rep_b': repetition_score(b),
        'rep_diff': repetition_score(a) - repetition_score(b),
        'sent_len_a': avg_sentence_len(a),
        'sent_len_b': avg_sentence_len(b),
        'code_a': float(len(CODE_BLOCK_RE.findall(a)) // 2),
        'code_b': float(len(CODE_BLOCK_RE.findall(b)) // 2),
        'bullets_a': float(len(BULLET_RE.findall(a))),
        'bullets_b': float(len(BULLET_RE.findall(b))),
        'numbered_a': float(len(NUMBERED_RE.findall(a))),
        'numbered_b': float(len(NUMBERED_RE.findall(b))),
        'headings_a': float(len(HEADING_RE.findall(a))),
        'headings_b': float(len(HEADING_RE.findall(b))),
        'refusal_a': float(len(REFUSAL_PATTERNS.findall(a))),
        'refusal_b': float(len(REFUSAL_PATTERNS.findall(b))),
        'hedge_a': float(len(HEDGE_PATTERNS.findall(a))),
        'hedge_b': float(len(HEDGE_PATTERNS.findall(b))),
    }


def features_to_prefix(f: dict[str, float]) -> str:
    return (
        f"[FEAT]"
        f" turns={f['turns']:.0f} pw={f['prompt_words']:.0f}"
        f" len_r={f['len_ratio']:.3f} wr={f['word_ratio']:.3f}"
        f" ovl_a={f['overlap_a']:.3f} ovl_b={f['overlap_b']:.3f} ovl_d={f['overlap_diff']:.3f}"
        f" sem_a={f['sem_a']:.3f} sem_b={f['sem_b']:.3f} sem_d={f['sem_diff']:.3f}"
        f" rep_a={f['rep_a']:.3f} rep_b={f['rep_b']:.3f} rep_d={f['rep_diff']:.3f}"
        f" code_a={f['code_a']:.0f} code_b={f['code_b']:.0f}"
        f" bul_a={f['bullets_a']:.0f} bul_b={f['bullets_b']:.0f}"
        f" num_a={f['numbered_a']:.0f} num_b={f['numbered_b']:.0f}"
        f" hd_a={f['headings_a']:.0f} hd_b={f['headings_b']:.0f}"
        f" ref_a={f['refusal_a']:.0f} ref_b={f['refusal_b']:.0f}"
        f" hed_a={f['hedge_a']:.0f} hed_b={f['hedge_b']:.0f}"
        f"[/FEAT]"
    )


def keep_head_tail(text: str, head: int = 260, tail: int = 200) -> str:
    words = text.split()
    if len(words) <= head + tail:
        return text
    return ' '.join(words[:head]) + ' [...] ' + ' '.join(words[-tail:])


def make_pair_text(prompt: str, a: str, b: str, feat: dict[str, float]) -> str:
    prefix = features_to_prefix(feat)
    p_c = keep_head_tail(prompt, head=280, tail=100)
    a_c = keep_head_tail(a, head=260, tail=200)
    b_c = keep_head_tail(b, head=260, tail=200)
    return (
        f"{prefix}\n"
        f"[PROMPT]\n{p_c}\n\n"
        f"[CANDIDATE_A]\n{a_c}\n\n"
        f"[CANDIDATE_B]\n{b_c}"
    )


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

similarity_vectorizer = build_similarity_vectorizer(train_df, test_df)
print('TF-IDF similarity vectorizer fitted.')

def map_winner_label(row: pd.Series) -> int:
    if int(row['winner_model_a']) == 1:
        return 0
    if int(row['winner_model_b']) == 1:
        return 1
    return 2


def prepare_base(df: pd.DataFrame, with_labels: bool, vec: TfidfVectorizer) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        prompt = '\n'.join(safe_load_list(row['prompt'])).strip()
        a = '\n'.join(safe_load_list(row['response_a'])).strip()
        b = '\n'.join(safe_load_list(row['response_b'])).strip()
        turns = max(len(safe_load_list(row['prompt'])),
                    len(safe_load_list(row['response_a'])),
                    len(safe_load_list(row['response_b'])))
        feat = extract_features(prompt, a, b, turns, vec)

        rec = {
            'id': row['id'],
            'prompt_text': prompt,
            'response_a_text': a,
            'response_b_text': b,
            'feat': feat,
        }
        if with_labels:
            rec['winner_label'] = map_winner_label(row)
        rows.append(rec)
    return pd.DataFrame(rows)


train_base = prepare_base(train_df, with_labels=True, vec=similarity_vectorizer)
test_base = prepare_base(test_df, with_labels=False, vec=similarity_vectorizer)

train_base.head(2)


In [ ]:
train_split, val_split = train_test_split(
    train_base,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_base['winner_label'],
)

print('Train split:', train_split.shape)
print('Val split:', val_split.shape)


def winner_to_soft_pref(label: int) -> float:
    if label == 0:
        return 1.0
    if label == 1:
        return 0.0
    return 0.5


def build_pairwise_df(df: pd.DataFrame, with_labels: bool, vec: TfidfVectorizer) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        p = row['prompt_text']
        a = row['response_a_text']
        b = row['response_b_text']

        # Original (A, B) features
        feat_ab = row['feat'] if isinstance(row['feat'], dict) \
            else extract_features(p, a, b, 1, vec)

        # Swapped (B, A) features — recompute so len_ratio etc. flip correctly
        turns = int(row['feat']['turns']) if isinstance(row['feat'], dict) else 1
        feat_ba = extract_features(p, b, a, turns, vec)

        rec_ab = {
            'id': row['id'],
            'pair_text': make_pair_text(p, a, b, feat_ab),
            'is_swapped': 0,
        }
        rec_ba = {
            'id': row['id'],
            'pair_text': make_pair_text(p, b, a, feat_ba),
            'is_swapped': 1,
        }

        if with_labels:
            y_ab = winner_to_soft_pref(int(row['winner_label']))
            rec_ab['labels'] = float(y_ab)
            rec_ba['labels'] = 1.0 - float(y_ab)

        rows.append(rec_ab)
        rows.append(rec_ba)

    return pd.DataFrame(rows)


train_pairwise = build_pairwise_df(train_split, with_labels=True, vec=similarity_vectorizer)
val_pairwise = build_pairwise_df(val_split, with_labels=True, vec=similarity_vectorizer)
test_pairwise = build_pairwise_df(test_base, with_labels=False, vec=similarity_vectorizer)

print('Train pairwise:', train_pairwise.shape)
print('Val pairwise:', val_pairwise.shape)
print('Test pairwise:', test_pairwise.shape)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_dataset(df: pd.DataFrame, with_labels: bool) -> Dataset:
    cols = ['pair_text', 'id', 'is_swapped']
    if with_labels:
        cols.append('labels')

    base = Dataset.from_pandas(df[cols].reset_index(drop=True), preserve_index=False)

    def tok(batch):
        return tokenizer(batch['pair_text'], truncation=True, max_length=MAX_LENGTH)

    tokenized = base.map(tok, batched=True)
    tokenized = tokenized.remove_columns(['pair_text'])
    return tokenized


train_ds = to_dataset(train_pairwise, with_labels=True)
val_ds = to_dataset(val_pairwise, with_labels=True)
test_ds = to_dataset(test_pairwise, with_labels=False)

train_ds

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    ignore_mismatched_sizes=True,
)

if torch.cuda.is_available():
    model = model.to('cuda')

model.config.pad_token_id = tokenizer.pad_token_id

class PairwiseTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels').float()
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(-1)
        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


import inspect

batch_size = TRAIN_BATCH_GPU
grad_accum_steps = max(1, TARGET_EFFECTIVE_BATCH // max(1, batch_size))

base_args = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=EVAL_BATCH_GPU,
    gradient_accumulation_steps=grad_accum_steps,
    eval_steps=400,
    save_strategy='steps',
    save_steps=400,
    logging_steps=50,
    load_best_model_at_end=True,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

supported = set(inspect.signature(TrainingArguments.__init__).parameters)
if 'eval_strategy' in supported:
    base_args['eval_strategy'] = 'steps'
elif 'evaluation_strategy' in supported:
    base_args['evaluation_strategy'] = 'steps'

args = TrainingArguments(**{k: v for k, v in base_args.items() if k in supported})

collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None)

trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)
trainer_supported = set(inspect.signature(Trainer.__init__).parameters)
if 'tokenizer' in trainer_supported:
    trainer_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in trainer_supported:
    trainer_kwargs['processing_class'] = tokenizer

trainer = PairwiseTrainer(**trainer_kwargs)

print('Gradient accumulation steps:', grad_accum_steps)
print('Effective train batch:', batch_size * grad_accum_steps)

trainer.train()

In [ ]:
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


# Validate and tune tie calibration parameters (alpha, beta).
val_pred = trainer.predict(val_ds)
val_logits = val_pred.predictions.reshape(-1)

val_meta = val_pairwise[['id', 'is_swapped']].copy().reset_index(drop=True)
val_meta['logit'] = val_logits

g = val_meta.pivot(index='id', columns='is_swapped', values='logit').reset_index()
g.columns = ['id', 'logit_ab', 'logit_ba']

# A-win probability with swap consistency averaging.
p_ab = sigmoid_np(g['logit_ab'].values)
p_a_from_ba = 1.0 - sigmoid_np(g['logit_ba'].values)
p_a = 0.5 * (p_ab + p_a_from_ba)

val_gold = val_split[['id', 'winner_label']].copy()
v = g.merge(val_gold, on='id', how='inner')
p_a = p_a[: len(v)]

best_alpha, best_beta = 0.15, 8.0
best_ll = float('inf')

alpha_grid = np.linspace(0.05, 0.45, 17)
beta_grid = np.linspace(2.0, 20.0, 19)

for alpha in alpha_grid:
    for beta in beta_grid:
        p_tie = alpha * np.exp(-beta * np.abs(p_a - 0.5))
        p_tie = np.clip(p_tie, 1e-6, 0.85)
        p_a3 = (1.0 - p_tie) * p_a
        p_b3 = (1.0 - p_tie) * (1.0 - p_a)

        probs = np.vstack([p_a3, p_b3, p_tie]).T
        probs = probs / probs.sum(axis=1, keepdims=True)

        ll = log_loss(v['winner_label'].values, probs, labels=[0, 1, 2])
        if ll < best_ll:
            best_ll = float(ll)
            best_alpha = float(alpha)
            best_beta = float(beta)

print(f'Best validation log_loss: {best_ll:.6f}')
print(f'Best tie alpha: {best_alpha:.3f}, beta: {best_beta:.3f}')

In [ ]:
test_pred = trainer.predict(test_ds)
test_logits = test_pred.predictions.reshape(-1)

test_meta = test_pairwise[['id', 'is_swapped']].copy().reset_index(drop=True)
test_meta['logit'] = test_logits

tg = test_meta.pivot(index='id', columns='is_swapped', values='logit').reset_index()
tg.columns = ['id', 'logit_ab', 'logit_ba']

p_ab = sigmoid_np(tg['logit_ab'].values)
p_a_from_ba = 1.0 - sigmoid_np(tg['logit_ba'].values)
p_a = 0.5 * (p_ab + p_a_from_ba)

p_tie = best_alpha * np.exp(-best_beta * np.abs(p_a - 0.5))
p_tie = np.clip(p_tie, 1e-6, 0.85)
p_a3 = (1.0 - p_tie) * p_a
p_b3 = (1.0 - p_tie) * (1.0 - p_a)

probs = np.vstack([p_a3, p_b3, p_tie]).T
probs = probs / probs.sum(axis=1, keepdims=True)

submission = pd.DataFrame({
    'id': tg['id'],
    'winner_model_a': probs[:, 0],
    'winner_model_b': probs[:, 1],
    'winner_tie': probs[:, 2],
})

submission.to_csv(SUBMISSION_PATH, index=False)
submission.head()

print('Saved:', SUBMISSION_PATH)
# 1.051324
# Best validation log_loss: 1.047191